In [ ]:
!pip install earthengine-api --upgrade
!pip install folium
!pip install pyproj

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 477.1/477.1 kB 6.7 MB/s eta 0:00:00
  Attempting uninstall: earthengine-api
    Found existing installation: earthengine-api 1.5.24
    Uninstalling earthengine-api-1.5.24:
      Successfully uninstalled earthengine-api-1.5.24


In [ ]:
import ee
import folium
ee.Authenticate()
ee.Initialize(project='ee-kulchiranut')

In [ ]:
!pip install globus-sdk
import geopandas as gpd
import json
from folium import plugins
import globus_sdk
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 414.6/414.6 kB 7.8 MB/s eta 0:00:00


In [ ]:
shp_paths = [
    "/content/NP.shp",
    "/content/pargkeaw.shp"
]

geodataframes_list = [gpd.read_file(f) for f in shp_paths]

all_shp = gpd.GeoDataFrame(pd.concat(geodataframes_list, ignore_index=True))

all_shp_wgs84 = all_shp.to_crs('EPSG:4326')

print(f"Loaded {len(shp_paths)} files.")
all_shp.head()

Loaded 2 files.


,AREA,Name,ตำบ,อำเ,จัง,ลำด,geometry
0,53.5952,เทศบาลเมืองหนองปรือ,ต.หนองปรือ,อ.หนองปรือ,จ.ชลบุรี,6.0000,"POLYGON Z ((709298.688 1434091.891 0, 709444.2..."
1,25.2279,เทศบาลเมืองบางแก้ว,ต.บางแก้ว,อ.บางพลี,จ.สมุทรปราการ,9.0000,"POLYGON Z ((676251.773 1507251.003 0, 676463.4..."


In [ ]:
all_shp.explore()

In [ ]:
def add_ee_layer(self, ee_object, vis_params, name):
    map_id_dict = None
    try:
        if isinstance(ee_object, ee.Image):
            map_id_dict = ee_object.getMapId(vis_params)
        elif isinstance(ee_object, ee.ImageCollection):
            map_id_dict = ee_object.mean().getMapId(vis_params)
        elif isinstance(ee_object, ee.FeatureCollection):
            styled_object = ee_object.style(**vis_params)
            map_id_dict = styled_object.getMapId()
        else:
            if hasattr(ee_object, 'getMapId'):
                map_id_dict = ee_object.getMapId(vis_params)
            else:
                raise TypeError(f"Unsupported Earth Engine object type.")

        if map_id_dict:
            folium.TileLayer(
                tiles=map_id_dict['tile_fetcher'].url_format,
                attr='Google Earth Engine',
                name=name,
                overlay=True,
                control=True
            ).add_to(self)
    except Exception as e:
        print(f"Could not add Earth Engine layer {name}: {e}")

folium.Map.add_ee_layer = add_ee_layer

In [ ]:
features = []
for i, row in all_shp_wgs84.iterrows():
    g = row['geometry']
    if g.geom_type == 'Polygon':
        features.append(ee.Feature(ee.Geometry.Polygon([[coord[0], coord[1]] for coord in g.exterior.coords]), row.drop('geometry').to_dict()))
    elif g.geom_type == 'MultiPolygon':
        polys = [ee.Geometry.Polygon([[coord[0], coord[1]] for coord in p.exterior.coords]) for p in g.geoms]
        features.append(ee.Feature(ee.Geometry.MultiPolygon(polys), row.drop('geometry').to_dict()))

area_ee = ee.FeatureCollection(features)

In [ ]:
import ee

area_ee_geometry = area_ee.geometry()

S2 = ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')\
.filterBounds(area_ee_geometry)\
.filterDate('2025-01-01','2025-03-01')

def maskcloud1(image):
    QA60 = image.select(['QA60'])
    return image.updateMask(QA60.lt(1))

S2 = S2.map(maskcloud1)
image = S2.median().select(['B2', 'B3', 'B4', 'B8'], ['blue', 'green', 'red', 'nir'])

ndvi = image.normalizedDifference(['nir', 'red']).rename('NDVI')
ndvi_clipped = ndvi.clip(area_ee_geometry)
ndvi_threshold = ndvi_clipped.updateMask(ndvi_clipped.gt(0.4))


vis_params = {
    'min': 0.0,
    'max': 1.0, 
    'palette': [
        'FFFFFF', 'CE7E45', 'DF923D', 'F18555', 'FCD163', '99B718', '74A901',
        '66A000', '529400', '3E8601', '207401', '056201', '004C00', '023801',
        '012E01', '011001', '011301'
     ],
}

vis_threshold = {
    'min': 0.4,
    'max': 1.0,
    'palette': ['green']
}


folium.Map.add_ee_layer = add_ee_layer

my_map = folium.Map(location=[13.7563, 100.5018], zoom_start=12)

my_map.add_ee_layer(ee.Feature(area_ee_geometry) , {'color': 'red'}, 'ขอบเขต')
my_map.add_ee_layer(ndvi_clipped, vis_params, 'NDVI')
my_map.add_ee_layer(ndvi_threshold, vis_threshold, 'ต้นไม้')

my_map.add_child(folium.LayerControl())

plugins.Fullscreen().add_to(my_map)

display(my_map)

In [ ]:
import pandas as pd

ndvi_pixel_area = ndvi_threshold.multiply(ee.Image.pixelArea()).rename('ndvi_area_m2')

def calculate_area(feat):
    poly_area = feat.area(maxError=1)

    stats = ndvi_pixel_area.reduceRegion(
        reducer=ee.Reducer.sum(),
        geometry=feat.geometry(),
        scale=10,
        maxPixels=1e9
    )

    ndvi_m2 = stats.get('ndvi_area_m2', ee.Number(0))

    pct_tree = ee.Number(ndvi_m2).divide(ee.Number(poly_area)).multiply(100)

    return feat.set({
        'polygon_area_m2': poly_area,
        'ndvi_area_m2': ndvi_m2,
        'pct_tree_cover': pct_tree
    })

computed_fc = area_ee.map(calculate_area)

features = computed_fc.getInfo()['features']
data = [f['properties'] for f in features]

df_area = pd.DataFrame(data)

df_area['polygon_area_km2'] = df_area['polygon_area_m2'] / 1e6
df_area['ndvi_area_km2'] = df_area['ndvi_area_m2'] / 1e6

cols = ['Name', 'polygon_area_m2', 'polygon_area_km2', 'ndvi_area_m2', 'ndvi_area_km2', 'pct_tree_cover']
df_area = df_area[cols]

total_m2 = df_area['polygon_area_m2'].sum()
total_ndvi_m2 = df_area['ndvi_area_m2'].sum()

pd.options.display.float_format = '{:,.4f}'.format

summary = pd.DataFrame([{
    'Name': 'รวมทั้งหมด',
    'polygon_area_m2': total_m2,
    'polygon_area_km2': total_m2 / 1e6,
    'ndvi_area_m2': total_ndvi_m2,
    'ndvi_area_km2': total_ndvi_m2 / 1e6,
    'pct_tree_cover': (total_ndvi_m2 / total_m2 * 100) if total_m2 > 0 else 0
}], index=['total'])

df_final = pd.concat([df_area, summary])

print(df_final.round(4))

df_final

                      Name  polygon_area_m2  polygon_area_km2    ndvi_area_m2  \
0      เทศบาลเมืองหนองปรือ  53,784,081.4493           53.7841  8,311,070.8839   
1       เทศบาลเมืองบางแก้ว  25,322,259.6460           25.3223  4,440,997.9829   
total           รวมทั้งหมด  79,106,341.0953           79.1063 12,752,068.8669   

       ndvi_area_km2  pct_tree_cover  
0             8.3111         15.4527  
1             4.4410         17.5379  
total        12.7521         16.1202  


,Name,polygon_area_m2,polygon_area_km2,ndvi_area_m2,ndvi_area_km2,pct_tree_cover
0,เทศบาลเมืองหนองปรือ,"53,784,081.4493",53.7841,"8,311,070.8839",8.3111,15.4527
1,เทศบาลเมืองบางแก้ว,"25,322,259.6460",25.3223,"4,440,997.9829",4.4410,17.5379
total,รวมทั้งหมด,"79,106,341.0953",79.1063,"12,752,068.8669",12.7521,16.1202


In [ ]:
from google.colab import files

files.download('Greenarea_summary.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import os
import geopandas as gpd

ndvi_binary_mask = ndvi_threshold

ndvi_vector = ndvi_binary_mask.reduceToVectors(
    geometry=area_ee_geometry,
    crs=ndvi_binary_mask.projection(),
    scale=10,
    geometryType='polygon',
    eightConnected=True,
    labelProperty='label'
)

ndvi_vector_simplified = ndvi_vector.map(lambda f: f.simplify(maxError=1))

output_dir = "exported_ndvi_shp"
if not os.path.exists(output_dir):
    os.makedirs(output_dir)

all_features = area_ee.getInfo()['features']
print(f"🚀 เริ่มต้นการส่งออกทั้งหมด {len(all_features)} พื้นที่...")

for feature in all_features:
    poly_name = feature['properties'].get('Name', f"area_{all_features.index(feature)}")

    try:
        poly_geom = ee.Feature(feature).geometry()
        specific_ndvi = ndvi_vector_simplified.filterBounds(poly_geom)

        count = specific_ndvi.size().getInfo()

        if count > 0:
            ndvi_data = specific_ndvi.getInfo()
            gdf_ndvi = gpd.GeoDataFrame.from_features(ndvi_data, crs="EPSG:4326")

            gdf_ndvi = gdf_ndvi[gdf_ndvi['label'] == 1]

            if not gdf_ndvi.empty:
                gdf_ndvi = gdf_ndvi.dissolve()

            if not gdf_ndvi.empty:
                clean_name = "".join(x for x in poly_name if x.isalnum() or x in "._-")
                file_path = os.path.join(output_dir, f"{clean_name}_green.shp")

                gdf_ndvi.to_file(file_path)
                print(f"✅ สำเร็จ: {poly_name} (พบ {len(gdf_ndvi)} รูปทรงสีเขียวหลังจากรวม)")
            else:
                print(f"⚪ ข้าม: {poly_name} (ไม่พบพื้นที่สีเขียวหลังจากกรองและรวม)")
        else:
            print(f"⚪ ข้าม: {poly_name} (ไม่พบพื้นที่สีเขียวในขอบเขต)")

    except Exception as e:
        print(f"❌ เกิดข้อผิดพลาดที่พื้นที่ {poly_name}: {e}")

print("\n✨ ประมวลผลเสร็จสมบูรณ์!")

🚀 เริ่มต้นการส่งออกทั้งหมด 2 พื้นที่...
❌ เกิดข้อผิดพลาดที่พื้นที่ เทศบาลเมืองหนองปรือ: Image.reduceToVectors: First band ('NDVI') of image must be integral.
❌ เกิดข้อผิดพลาดที่พื้นที่ เทศบาลเมืองบางแก้ว: Image.reduceToVectors: First band ('NDVI') of image must be integral.

✨ ประมวลผลเสร็จสมบูรณ์!


In [ ]:
import shutil
from google.colab import files

zip_filename = 'green_areas.zip'

shutil.make_archive('green_areas', 'zip', output_dir)

files.download(f"{zip_filename}")

print(f"📦 บีบอัดและเตรียมดาวน์โหลดไฟล์ {zip_filename} เรียบร้อยแล้ว!")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

📦 บีบอัดและเตรียมดาวน์โหลดไฟล์ green_areas.zip เรียบร้อยแล้ว!
